In [0]:
from pyspark.sql.functions import *

# ==========================================
# S3 PATHS
# ==========================================

incoming_path = "s3://workflowautomationgiri/incoming/"
processed_path = "s3://workflowautomationgiri/processed/"

# ==========================================
# GET JSON FILES
# ==========================================

all_files = dbutils.fs.ls(incoming_path)

json_files = [
    file.path
    for file in all_files
    if file.path.endswith(".json")
]

# ==========================================
# READ PROCESSED LOG
# ==========================================

processed_df = spark.table(
    "workauto.gold.processed_files_log"
).filter(
    col("file_type") == "json"
)

processed_files = [
    row["file_name"]
    for row in processed_df.collect()
]

# ==========================================
# FILTER NEW FILES
# ==========================================

new_files = []

for file in json_files:

    file_name = file.split("/")[-1]

    if file_name not in processed_files:
        new_files.append(file)

# ==========================================
# PROCESS FILES
# ==========================================

if len(new_files) > 0:

    json_df = (
        spark.read
        .format("json")
        .option("multiline", "true")
        .load(new_files)
    )

    json_df = (
        json_df
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "ingestion_time",
            current_timestamp()
        )
    )

    (
        json_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.bronze.json_raw")
    )

    # UPDATE LOG
    processed_log_df = spark.createDataFrame(
        [
            (file.split("/")[-1], "json")
            for file in new_files
        ],
        ["file_name", "file_type"]
    ).withColumn(
        "processed_time",
        current_timestamp()
    )

    (
        processed_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.gold.processed_files_log")
    )

    # MOVE FILES
    for file in new_files:

        file_name = file.split("/")[-1]

        dbutils.fs.mv(
            file,
            processed_path + file_name
        )

    print("JSON Files Processed Successfully")

else:

    print("No New JSON Files Found")